## Organized reproduction copy

Run the setup cell first, then Run All. Original analysis cells and saved UCLA outputs below are unchanged. The setup only resolves the organized folder paths. Inputs are linked from `04_results/reproduction_inputs`; new runs write to `04_results/reruns`, preserving the supplied results. Notebook 01 reconstructs the feature table; notebooks 02–03 use the included validated table. Older mapping-unverified wording is historical: all 2,661 mappings were subsequently verified in notebook 03.


In [ ]:
# Folder setup only; the original analysis cells below are unchanged.
from pathlib import Path
import os
_here = Path.cwd().resolve()
_project = next((p for p in [_here, *_here.parents] if (p/'01_original_data').is_dir() and (p/'02_notebooks').is_dir() and (p/'04_results').is_dir()), None)
if _project is None:
    raise FileNotFoundError('Start Jupyter inside the organized wallet_profit_research folder.')
os.chdir(_project/'02_notebooks')
os.environ['WALLET_MODEL_ROOT'] = str(Path.cwd())
os.environ['WALLET_VALIDATION_ROOT'] = str(Path.cwd())
os.environ['WALLET_MODEL_MODE'] = 'production'
os.environ['WALLET_VALIDATION_MODE'] = 'production'
os.environ['WALLET_MODEL_OUTPUT'] = str(_project/'04_results/reruns/modeling/production')
os.environ['WALLET_VALIDATION_OUTPUT'] = str(_project/'04_results/reruns/validation/production')
assert (Path('data')/'wallet_market_candidates_full_week.parquet').is_file()
assert len(list((Path('data')/'weekly_trades').glob('trades-*.parquet'))) == 7
print('Inputs:', Path('data').resolve())
print('New outputs:', (_project/'04_results/reruns').resolve())


# 结算证据、嵌套经验诊断与严格24小时历史

本轮延续已完成的条件盈利项目，检验结算字段与具体token/结果的证据，比较同一批钱包–市场行上的“只加一个历史指标”和“同时加三个”，并从全部七个原始逐笔文件构建严格预测前24h特征。不是为了把p值变小，也不会根据预测效果挑结算映射或窗口。

每个钱包–市场只有一个预测点：提供的一周内首次观察时间戳整批成交后，不是终身第一次入场。条件毛利=卖出金额−买入金额+Σ(净买入份额×结果payout)，主映射假设outcome0 payout=price_at_close、outcome1为补数；外部验证只对实际匹配的市场/token成立。标签Y=1[毛利>1e−6]，平手/非正为0；负观察库存标签未知。期初成本、费用和非交易持仓变化仍未齐，即使结果映射被核验，真实净利润仍未知。

严格24h范围是`[T−24h,T)`，排除整个目标市场与所有T同时间戳。完整提供文件跨度需要T>=Sep30UTC；更早行保留缺口标记，不填0当完整历史。原始输入和Round04结果只读。原Round04测试已经看过，本轮所有预测比较均明确为探索性复用，不是新鲜独立测试。

运行：`WALLET_VALIDATION_MODE=production WALLET_VALIDATION_ROOT=$PWD python execute_validation_notebook.py`。默认核`walletvalidation`，可用`WALLET_VALIDATION_KERNEL`覆盖；输出`results/validation/production/`，`WALLET_VALIDATION_OUTPUT`可指定路径。所有运算均内嵌，无外部分析脚本依赖。


## 输入与独立映射证据

保留证据原始状态和hash；链上验证需要精确元数据身份、derived tokenID与已定payout向量对应，不把当前价格/内部布尔一致当证明。没有证据覆盖的行保持未核验；本轮不静默换标签。

In [ ]:
import os,sys,json,time,hashlib,socket,platform,re,warnings,gc
from pathlib import Path
from collections import deque
import numpy as np,pandas as pd,pyarrow,duckdb,scipy,sklearn,statsmodels
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss,brier_score_loss,roc_auc_score,average_precision_score
from statsmodels.stats.multitest import multipletests
from IPython.display import display,Markdown
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1);pyarrow.set_cpu_count(1);pyarrow.set_io_thread_count(1)
START=time.perf_counter()
ROOT=Path(os.environ.get('WALLET_VALIDATION_ROOT','.')).resolve()
MODE=os.environ.get('WALLET_VALIDATION_MODE','production')
OUT=Path(os.environ.get('WALLET_VALIDATION_OUTPUT',str(ROOT/'results/validation'/MODE))).resolve()
for protected in [ROOT/'results/modeling',ROOT/'results/full_week',ROOT/'deliverables/modeling',ROOT/'deliverables/full_week']:
 if OUT==protected.resolve() or protected.resolve() in OUT.parents:raise RuntimeError('Refusing to write into preserved earlier-round artifacts: '+str(OUT))
OUT.mkdir(parents=True,exist_ok=True)
CONFIG={'mode':MODE,'seed':20261019,'bootstrap_replicates':20 if MODE=='smoke' else 300,
 'logistic_C':1.,'logistic_max_iter':500,'classification_threshold':.5,'memory_limit':'3GB','threads':1,
 'window_hours':24,'week_start':'2026-09-29T00:00:00Z','full_window_start':'2026-09-30T00:00:00Z',
 'validation_start':'2026-10-03T00:00:00Z','test_start':'2026-10-04T00:00:00Z','float_tolerance':1e-8,
 'evaluation_status':'Previously inspected Round04 test reused; exploratory robustness, not fresh held-out confirmation'}
def sha(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
 return h.hexdigest()
def find_input(name):
 for folder in [ROOT/'data',ROOT/'server_run/round03',ROOT/'results/full_week']:
  if (folder/name).exists():return folder/name
 raise FileNotFoundError(name)
input_path=find_input('wallet_market_candidates_full_week.parquet');metadata_path=find_input('weekly_market_metadata_complete.csv')
d=pd.read_parquet(input_path);metadata=pd.read_csv(metadata_path)
assert len(d)==96184 and d.conditional_profit_label.notna().sum()==89736 and d.confirmed_net_profit_label.isna().all()
assert not d[['wallet','market']].duplicated().any()
for c in ['prediction_time','feature_week_history_first_time','feature_week_history_latest_time']:d[c]=pd.to_datetime(d[c],utc=True)
metadata['resolved_at']=pd.to_datetime(metadata.resolved_at,utc=True,format='mixed',errors='coerce')
provenance={'config':CONFIG,'inputs':{str(input_path):sha(input_path),str(metadata_path):sha(metadata_path)},
 'versions':{'python':sys.version,'numpy':np.__version__,'pandas':pd.__version__,'pyarrow':pyarrow.__version__,
 'duckdb':duckdb.__version__,'scipy':scipy.__version__,'sklearn':sklearn.__version__,'statsmodels':statsmodels.__version__},
 'hostname':socket.gethostname(),'scheduler_job_id':os.environ.get('JOB_ID'),'platform':platform.platform()}
# Main stages independent public evidence; no mapping is inferred from fit or current prices.
mapping_path=ROOT/'data/settlement_mapping_audit.csv'
if not mapping_path.exists():mapping_path=ROOT/'data/public_mapping_evidence.csv'
if mapping_path.exists():
 mapping_evidence=pd.read_csv(mapping_path);provenance['inputs'][str(mapping_path)]=sha(mapping_path)
else:
 mapping_evidence=pd.DataFrame({'market':d.market.unique(),'verification_status':'unresolved_no_independent_evidence_staged'})
mapping_summary_path=ROOT/'data/settlement_mapping_summary.json'
if mapping_summary_path.exists():provenance['inputs'][str(mapping_summary_path)]=sha(mapping_summary_path)
# Preserve original evidence schema/status rather than silently upgrading any claim.
mapping_evidence.to_csv(OUT/'public_mapping_evidence.csv',index=False)
public_raw=ROOT/'data/settlement_evidence'
if public_raw.exists():provenance['public_raw_hashes']={str(p.relative_to(ROOT)):sha(p) for p in public_raw.rglob('*') if p.is_file()}
display(mapping_evidence.head(12))
print('映射证据按实际覆盖列出；本轮不改主标签映射，也不按预测效果挑映射。所有真实净标签仍未知。')


## 有界内存的严格24h统计

DuckDB单线程3GB先按钱包–市场–时间聚合，再按钱包/时间流式读取。滑动deque只保留24h；在整批T加入前发出快照，下界只删小于T−24h的记录。原始目标市场最早时间与全部96,184个T逐一核对，因此在首次提供周目标入场之前，根本没有该目标市场历史；全局严格前缀在这些特定预测点等价于显式排除目标。其他预测点不能使用这个等价性。

HHI分子在每次发出快照时重新由市场BUY金额计算，避免删除浮点漂移；平均ticket按BUY行数，包含0美元BUY，不能把无BUY行与无BUY美元混为一谈。

In [ ]:
"""Bounded streaming strict24h histories, usable inline in a notebook."""
def stream_fixed_histories(records,entries,week_start_ms,window_ms=86400000,float_tolerance=1e-8):
    from collections import deque
    import math
    output=[];max_square_drift=0.;last_wallet=None;last_time=None
    for wallet,market,t,n,nb,b in records:
        t=int(t);n=int(n);nb=int(nb);b=float(b)
        if wallet!=last_wallet:
            queue=deque();market_rows={};market_buy={};market_buy_rows={};time_counts={}
            total_rows=buy_rows=0;buy_dollars=0.;square=0.
            cumulative_rows=cumulative_buy_rows=cumulative_timestamps=0;cumulative_dollars=0.;cumulative_markets=set()
            last_wallet=wallet;last_time=None
        if t!=last_time:
            # Lower endpoint INCLUDED: remove only records strictly older than T-24h.
            while queue and queue[0][0]<t-window_ms:
                ot,om,on,onb,ob=queue.popleft()
                total_rows-=on;buy_rows-=onb;buy_dollars-=ob
                old=market_buy.get(om,0.);new=old-ob
                square+=new*new-old*old
                market_rows[om]-=on;market_buy_rows[om]-=onb
                if market_buy_rows[om]==0:new=0.
                if market_rows[om]==0:
                    market_rows.pop(om);market_buy.pop(om,None);market_buy_rows.pop(om,None)
                else:
                    tolerance=float_tolerance*max(1.,abs(old),abs(ob))
                    assert new>=-tolerance,('negative_buy_sum',wallet,om,new)
                    market_buy[om]=0. if -tolerance<=new<0 else new
                time_counts[ot]-=1
                if time_counts[ot]==0:time_counts.pop(ot)
            if total_rows==0:
                # Exact empty state, not a zero-fill assumption about missing pre-week history.
                buy_dollars=0.;square=0.;assert buy_rows==0 and not market_rows and not time_counts
            elif buy_dollars<0:
                assert abs(buy_dollars)<=float_tolerance*max(1.,cumulative_dollars)
                buy_dollars=0.
            ids=entries.get(wallet,{}).get(t,[])
            if ids:
                # Recompute HHI numerator at every emitted cutoff to prevent subtraction drift.
                exact_square=sum(value*value for value in market_buy.values())
                exact_dollars=sum(market_buy.values())
                assert abs(exact_dollars-buy_dollars)<=float_tolerance*max(1.,cumulative_dollars)
                buy_dollars=exact_dollars
                max_square_drift=max(max_square_drift,abs(exact_square-square)/max(1.,exact_square))
                assert abs(exact_square-square)<=float_tolerance*max(1.,exact_square,cumulative_dollars*cumulative_dollars)
                square=exact_square
                hhi=exact_square/(buy_dollars*buy_dollars) if buy_dollars>0 else math.nan
                if buy_dollars>0:assert -float_tolerance<=hhi<=1+float_tolerance,(wallet,t,hhi)
                full=t-window_ms>=week_start_ms
                row={'full24h_observed_span':int(full),'week_buy_fill_rows':cumulative_buy_rows,
                     'week_avg_buy_ticket':cumulative_dollars/cumulative_buy_rows if cumulative_buy_rows else math.nan,
                     'proof_week_fill_rows':cumulative_rows,'proof_week_markets':len(cumulative_markets),
                     'proof_week_buy_dollars':cumulative_dollars,'proof_week_distinct_timestamps':cumulative_timestamps}
                stats={'fill_rows':total_rows,'distinct_timestamps':len(time_counts),'markets':len(market_rows),
                    'buy_fill_rows':buy_rows,'buy_dollars':buy_dollars,'buy_market_hhi':hhi,
                    'avg_buy_ticket':buy_dollars/buy_rows if buy_rows else math.nan,
                    'no_observed_rows':int(total_rows==0),'no_buy_rows':int(buy_rows==0),'no_buy_dollars':int(buy_dollars==0)}
                for key,value in stats.items():row['fixed24h_'+key]=value if full else math.nan
                for pair_id in ids:output.append({'pair_id':pair_id,**row})
            # Emit BEFORE any rows in the entire T batch, even across markets.
            cumulative_timestamps+=1;last_time=t
        old=market_buy.get(market,0.);new=old+b;square+=new*new-old*old
        market_buy[market]=new;market_rows[market]=market_rows.get(market,0)+n;market_buy_rows[market]=market_buy_rows.get(market,0)+nb
        time_counts[t]=time_counts.get(t,0)+1
        queue.append((t,market,n,nb,b));total_rows+=n;buy_rows+=nb;buy_dollars+=b
        cumulative_rows+=n;cumulative_buy_rows+=nb;cumulative_dollars+=b;cumulative_markets.add(market)
    return output,{'max_relative_maintained_square_drift':max_square_drift,'emitted_rows':len(output),
                   'float_tolerance':float_tolerance,'HHI_numerator_recomputed_at_every_emitted_cutoff':True}

def fixed_history_fixture():
    import math
    day=86400000
    # At T=2days: lower boundary1day included, just-before boundary excluded, allT ties excluded.
    records=[('w','old',day-1,1,1,9.),('w','a',day,1,1,2.),
             ('w','b',day+1,1,0,0.),('w','target',2*day,1,1,99.),
             ('w','tied_other',2*day,1,1,50.),('z','zero',day,1,1,0.),('z','target',2*day,1,1,5.)]
    entries={'w':{2*day:[0]},'z':{2*day:[1]}}
    rows,_=stream_fixed_histories(records,entries,0)
    first,zero=rows
    assert first['fixed24h_fill_rows']==2 and first['fixed24h_markets']==2
    assert first['fixed24h_buy_dollars']==2 and first['fixed24h_buy_market_hhi']==1
    assert first['fixed24h_buy_fill_rows']==1 and first['fixed24h_avg_buy_ticket']==2
    assert zero['fixed24h_buy_fill_rows']==1 and zero['fixed24h_avg_buy_ticket']==0
    assert math.isnan(zero['fixed24h_buy_market_hhi']) and zero['fixed24h_no_buy_rows']==0 and zero['fixed24h_no_buy_dollars']==1
    early,_=stream_fixed_histories([('w','target',1,1,1,1.)],{'w':{1:[0]}},0)
    assert early[0]['full24h_observed_span']==0 and math.isnan(early[0]['fixed24h_fill_rows'])
    empty,_=stream_fixed_histories([('w','target',day,1,1,1.)],{'w':{day:[0]}},0)
    assert empty[0]['fixed24h_fill_rows']==0 and math.isnan(empty[0]['fixed24h_avg_buy_ticket'])
    print('Strict24h fixture passed: lower inclusion, upper ties, target exclusion, sell breadth, zero-dollar BUY, truncation and empty history.')

fixed_history_fixture()
RAW_START=time.perf_counter()
files=sorted((ROOT/'data/weekly_trades').glob('trades-*.parquet'));assert len(files)==7
raw_manifest=ROOT/'data/weekly_trades/download_manifest.json'
if raw_manifest.exists():provenance['inputs'][str(raw_manifest)]=sha(raw_manifest)
provenance['raw_files']=[{'path':str(p),'bytes':p.stat().st_size,'sha256':sha(p)} for p in files]
spill=OUT/'duckdb_spill';spill.mkdir(exist_ok=True)
con=duckdb.connect();con.execute('SET threads=1');con.execute("SET memory_limit='3GB'");con.execute("SET TimeZone='UTC'")
con.execute("SET temp_directory='"+str(spill).replace("'","''")+"'")
con.from_parquet([str(p) for p in files]).create_view('raw_week')
assert con.execute('SELECT count(*) FROM raw_week').fetchone()[0]==6815814
con.register('candidate_markets',pd.DataFrame({'market':d.market.unique()}))
con.register('entry_keys',d[['wallet','market','prediction_time']])
raw_first=con.execute('''SELECT r.wallet,r.market,min(r.time) raw_first FROM raw_week r
 JOIN candidate_markets c USING(market) GROUP BY r.wallet,r.market''').df()
raw_first['raw_first']=pd.to_datetime(raw_first.raw_first,utc=True)
first_check=d[['wallet','market','prediction_time']].merge(raw_first,on=['wallet','market'],validate='one_to_one')
assert len(first_check)==96184 and first_check.prediction_time.eq(first_check.raw_first).all()
# Therefore before each T no same-wallet target-market fill exists anywhere in supplied files.
con.register('candidate_wallets',pd.DataFrame({'wallet':d.wallet.unique()}))
con.execute('''CREATE TABLE history_batches AS SELECT r.wallet,r.market,epoch_ms(r.time)::BIGINT t,
 count(*)::BIGINT n,count(*) FILTER(WHERE side='BUY')::BIGINT nb,
 sum(CASE WHEN side='BUY' THEN dollars ELSE 0 END)::DOUBLE b
 FROM raw_week r JOIN candidate_wallets c USING(wallet) GROUP BY r.wallet,r.market,r.time''')
entry_specs={}
for i,row in d[['wallet','prediction_time']].iterrows():entry_specs.setdefault(row.wallet,{}).setdefault(int(row.prediction_time.value//1000000),[]).append(i)
def batch_records():
 reader=con.execute('SELECT wallet,market,t,n,nb,b FROM history_batches ORDER BY wallet,t,market').fetch_record_batch(rows_per_batch=65536)
 for batch in reader:
  values=batch.to_pydict()
  yield from zip(values['wallet'],values['market'],values['t'],values['n'],values['nb'],values['b'])
window_rows,window_diagnostics=stream_fixed_histories(batch_records(),entry_specs,int(pd.Timestamp(CONFIG['week_start']).value//1000000),float_tolerance=CONFIG['float_tolerance'])
w=pd.DataFrame(window_rows).set_index('pair_id').sort_index();assert list(w.index)==list(d.index)
for source,target in [('proof_week_fill_rows','feature_week_history_fill_rows'),('proof_week_markets','feature_week_history_markets'),
 ('proof_week_buy_dollars','feature_week_history_buy_dollars'),('proof_week_distinct_timestamps','feature_week_history_distinct_timestamps')]:
 assert np.allclose(w[source],d[target],atol=1e-6,rtol=1e-9),(source,target)
d['audit_only_fixed24h_left_truncated']=1-w.full24h_observed_span
for c in w:
 if c.startswith('fixed24h_'):d['feature_'+c]=w[c]
d['feature_week_history_buy_fill_rows']=w.week_buy_fill_rows
d['feature_week_history_avg_buy_ticket']=w.week_avg_buy_ticket
for period in ['fixed24h','week_history']:
 for c in ['fill_rows','markets','buy_dollars','avg_buy_ticket']:
  d['feature_'+period+'_log_'+c]=np.log1p(d['feature_'+period+'_'+c])
assert d.loc[d.audit_only_fixed24h_left_truncated.eq(1),'feature_fixed24h_fill_rows'].isna().all()
window_diagnostics['elapsed_seconds']=time.perf_counter()-RAW_START
print('固定24h流式构建:',window_diagnostics)


## 独立直接范围过滤

另一路DuckDB显式按每个T过滤时间上下界与目标市场，校验全部2,462个试点和128个确定性非试点样本，包括缺失状态。这里不复用滑动deque统计。

In [ ]:
pilot_path=ROOT/'data/pilot_market_metadata.csv'
if pilot_path.exists():pilot_markets=set(pd.read_csv(pilot_path).market)
else:pilot_markets={'cs2-g2-1win-2026-10-04','highest-temperature-in-tokyo-on-october-2-2026-23c','btc-updown-5m-1791162900'}
pilot_ids=d.index[d.market.isin(pilot_markets)].to_numpy();assert len(pilot_ids)==2462
nonpilot=d[~d.market.isin(pilot_markets)&d.audit_only_fixed24h_left_truncated.eq(0)].copy()
nonpilot['stable_sample_hash']=(nonpilot.wallet+'|'+nonpilot.market).map(lambda x:hashlib.sha256(x.encode()).hexdigest())
verify_ids=np.r_[pilot_ids,nonpilot.sort_values('stable_sample_hash').index[:128]]
verify_keys=d.loc[verify_ids,['wallet','market','prediction_time']].copy().reset_index().rename(columns={'index':'pair_id','market':'target_market','prediction_time':'cutoff'})
con.register('verify_keys',verify_keys)
con.execute('''CREATE TABLE verify_history AS SELECT c.pair_id,r.* FROM raw_week r JOIN verify_keys c ON r.wallet=c.wallet
 WHERE r.time<c.cutoff AND r.time>=c.cutoff-INTERVAL '24 HOURS' AND r.market<>c.target_market''')
independent=con.execute('''WITH totals AS (SELECT pair_id,count(*) fill_rows,count(DISTINCT time) distinct_timestamps,
 count(DISTINCT market) markets,count(*) FILTER(WHERE side='BUY') buy_fill_rows,
 sum(CASE WHEN side='BUY' THEN dollars ELSE 0 END) buy_dollars FROM verify_history GROUP BY pair_id),
 bm AS (SELECT pair_id,market,sum(dollars) b FROM verify_history WHERE side='BUY' GROUP BY pair_id,market),
 square AS (SELECT pair_id,sum(b*b) s FROM bm GROUP BY pair_id)
 SELECT t.*,s.s/(t.buy_dollars*t.buy_dollars) buy_market_hhi,t.buy_dollars/t.buy_fill_rows avg_buy_ticket FROM totals t LEFT JOIN square s USING(pair_id)''').df().set_index('pair_id').reindex(verify_ids)
for c in ['fill_rows','distinct_timestamps','markets','buy_fill_rows','buy_dollars']:independent[c]=independent[c].fillna(0)
independent.loc[independent.buy_dollars.eq(0),'buy_market_hhi']=np.nan
independent.loc[independent.buy_fill_rows.eq(0),'avg_buy_ticket']=np.nan
verified_full=d.loc[verify_ids].audit_only_fixed24h_left_truncated.eq(0)
assert (verified_full.to_numpy()==d.loc[verify_ids].prediction_time.ge(pd.Timestamp(CONFIG['full_window_start'])).to_numpy()).all()
assert d.loc[verify_ids[2462:],'audit_only_fixed24h_left_truncated'].eq(0).all()
assert d.loc[verify_ids[~verified_full.to_numpy()],'feature_fixed24h_fill_rows'].isna().all()
for c in independent:
 actual=d.loc[verify_ids,'feature_fixed24h_'+c]
 assert np.allclose(actual[verified_full],independent.loc[verified_full,c],equal_nan=True,atol=1e-6,rtol=1e-8),c
verification_report={'all_pilot_cutoffs':2462,'nonpilot_cutoffs':128,'full24h_cutoffs_compared':int(verified_full.sum()),'nonpilot_full24h_compared':128,
 'early_pilot_unavailable_coverage_flags_checked':int((~verified_full).sum()),
 'independent_SQL_explicit_target_exclusion':True,'all_passed':True}
print('独立直接SQL范围过滤校验:',verification_report)
con.close();con=None
import shutil
shutil.rmtree(spill,ignore_errors=True)
d[[c for c in d if c in ['wallet','market','prediction_time','audit_only_fixed24h_left_truncated'] or c.startswith('feature_fixed24h_') or c in ['feature_week_history_buy_fill_rows','feature_week_history_avg_buy_ticket','feature_week_history_log_avg_buy_ticket']]].to_parquet(OUT/'fixed24h_history_features.parquet',index=False)
del window_rows,entry_specs,first_check,raw_first,nonpilot,w;gc.collect()


## 官方事件链接、原时间分割与完整24h共同人群

原保守事件组在共享独立匹配官方事件ID时只合并不拆细，防止已知官方关联跨分割；不把crypto同日组拆成较细官方事件。保留原分组兼容计数。按所有输入时点先分配晚时段测试组，早成员丢弃；训练标签须在评估前按记录结算。记录结算时间仍不是实际结果公开时间。

所有嵌套及累计/24h预测使用完全相同完整24h行，明确列出入选/因早期截断移除数量。

In [ ]:
def event_proxy(slug):
    weather=re.match(r'((?:highest|lowest)-temperature-in-.+?-on-[a-z]+-\d+-\d{4})(?:-|$)',slug)
    if weather:return weather[1],'weather_city_date_proxy'
    dated=re.match(r'(.+?-\d{4}-\d{2}-\d{2})(?:-|$)',slug)
    if dated:return dated[1],'dated_match_prefix_proxy'
    crypto=re.match(r'([a-z]+)-updown-(?:5m|15m|1h|4h)-(\d+)$',slug)
    if crypto:return 'crypto-day-'+str(int(crypto[2])//86400),'all_asset_crypto_UTC_day_proxy'
    threshold=re.match(r'(.+?)-above-\d+(-on-.+)$',slug)
    if threshold:return threshold[1]+threshold[2],'asset_time_threshold_proxy'
    return slug,'unmapped_market_as_proxy'
events=pd.DataFrame([{'market':m,'event_group':event_proxy(m)[0],'event_mapping_rule':event_proxy(m)[1]} for m in d.market.unique()])
assert not events.market.duplicated().any()
# Preserve conservative proxy families; merge them only when independently matched official event links require it.
def as_bool(value):return str(value).strip().lower() in ['true','1','yes']
def parse_event_ids(row):
 raw=row.get('official_event_ids',None)
 if raw is not None and not (isinstance(raw,float) and np.isnan(raw)):
  try:
   parsed=json.loads(raw) if isinstance(raw,str) else raw
   if isinstance(parsed,list):return [str(x) for x in parsed if x is not None]
  except (ValueError,TypeError):pass
 raw=row.get('official_event_id',None)
 return [] if raw is None or str(raw).strip().lower() in ['nan','none',''] else [str(raw)]
identity_field='local_identity_matches' if 'local_identity_matches' in mapping_evidence else ('metadata_identity_match' if 'metadata_identity_match' in mapping_evidence else 'onchain_verified')
official_links={}
for _,record in mapping_evidence.iterrows():
 if 'market' in record and identity_field in record and as_bool(record[identity_field]):
  official_links[record.market]=parse_event_ids(record)
parent={group:group for group in events.event_group}
def find_group(group):
 while parent[group]!=group:parent[group]=parent[parent[group]];group=parent[group]
 return group
def union_group(a,b):
 a=find_group(a);b=find_group(b)
 if a!=b:parent[max(a,b)]=min(a,b)
owner={}
for record in events.itertuples(index=False):
 for identity in official_links.get(record.market,[]):
  if identity in owner:union_group(record.event_group,owner[identity])
  else:owner[identity]=record.event_group
old_events=events.copy().rename(columns={'event_group':'round04_event_proxy'})
events['event_group']=events.event_group.map(find_group)
events['official_event_ids']=events.market.map(lambda market:json.dumps(official_links.get(market,[])))
events=events.merge(old_events[['market','round04_event_proxy']],on='market',validate='one_to_one')
official_linkage_summary={'identity_field_used':identity_field,'markets_with_matched_official_ids':sum(bool(v) for v in official_links.values()),
 'original_proxy_groups':old_events.round04_event_proxy.nunique(),'official_connected_proxy_groups':events.event_group.nunique(),
 'changed_market_group_labels':int(events.event_group.ne(events.round04_event_proxy).sum()),
 'rule':'Union full conservative proxy groups sharing any exact-identity matched official event ID; never split proxies into finer IDs'}
events.to_csv(OUT/'official_event_connected_mapping.csv',index=False)
# Old Round04 cohort counts are compatibility evidence only; all primary new fits use connected groups.
compat=d[['wallet','market','prediction_time','conditional_profit_label']].merge(old_events[['market','round04_event_proxy']],on='market').merge(metadata[['market','resolved_at']],on='market')
compat_test=set(compat.loc[compat.prediction_time.ge(pd.Timestamp(CONFIG['test_start'])),'round04_event_proxy'])
compat_val=set(compat.loc[compat.prediction_time.ge(pd.Timestamp(CONFIG['validation_start']))&compat.prediction_time.lt(pd.Timestamp(CONFIG['test_start']))&~compat.round04_event_proxy.isin(compat_test),'round04_event_proxy'])
compat['old_partition']='train';compat.loc[compat.round04_event_proxy.isin(compat_val),'old_partition']='validation';compat.loc[compat.round04_event_proxy.isin(compat_test),'old_partition']='test'
compat.groupby('old_partition').agg(all_rows=('wallet','size'),markets=('market','nunique')).to_csv(OUT/'round04_proxy_assignment_compatibility_counts.csv')
print('官方事件链接审计（不把保守crypto日组拆细）:',official_linkage_summary)

d=d.merge(events,on='market',validate='many_to_one').merge(metadata[['market','resolved_at']],on='market',validate='many_to_one')
T0=pd.Timestamp(CONFIG['validation_start']);T1=pd.Timestamp(CONFIG['test_start'])
# Assignment depends on event IDs and all observed entry times, never on Y or profitability eligibility.
test_events=set(d.loc[d.prediction_time.ge(T1),'event_group'])
validation_events=set(d.loc[d.prediction_time.ge(T0)&d.prediction_time.lt(T1)&~d.event_group.isin(test_events),'event_group'])
d['assigned_partition']='development_train'
d.loc[d.event_group.isin(validation_events),'assigned_partition']='validation'
d.loc[d.event_group.isin(test_events),'assigned_partition']='test'
d['eligible_conditional']=d.conditional_profit_label.notna()&d.resolved_at.notna()&d.prediction_time.lt(d.resolved_at)
d['split']='excluded'
d.loc[d.eligible_conditional&d.assigned_partition.eq('development_train')&d.prediction_time.lt(T0)&d.resolved_at.lt(T0),'split']='train'
d.loc[d.eligible_conditional&d.assigned_partition.eq('validation')&d.prediction_time.ge(T0)&d.prediction_time.lt(T1)&d.resolved_at.lt(T1),'split']='validation'
d.loc[d.eligible_conditional&d.assigned_partition.eq('test')&d.prediction_time.ge(T1),'split']='test'
d['final_refit']=d.eligible_conditional&~d.event_group.isin(test_events)&d.prediction_time.lt(T1)&d.resolved_at.lt(T1)
d['exclusion_reason']='retained'
d.loc[~d.conditional_profit_label.notna(),'exclusion_reason']='conditional_label_unknown_negative_inventory'
d.loc[d.conditional_profit_label.notna()&d.resolved_at.isna(),'exclusion_reason']='recorded_resolution_missing'
d.loc[d.conditional_profit_label.notna()&d.resolved_at.notna()&d.prediction_time.ge(d.resolved_at),'exclusion_reason']='entry_at_or_after_recorded_resolution'
d.loc[d.eligible_conditional&d.split.eq('excluded'),'exclusion_reason']='preboundary_member_of_heldout_event_or_training_label_not_mature'
train=d[d.split.eq('train')].copy();val=d[d.split.eq('validation')].copy();test=d[d.split.eq('test')].copy();refit=d[d.final_refit].copy()
assert len(train)>0 and len(val)>0 and len(test)>0 and len(refit)>0
for left,right in [(train,val),(train,test),(val,test),(refit,test)]:
    assert not set(left.event_group)&set(right.event_group)
    assert not set(left.market)&set(right.market)
assert train.prediction_time.max()<T0 and train.resolved_at.max()<T0
assert val.prediction_time.min()>=T0 and val.prediction_time.max()<T1 and val.resolved_at.max()<T1
assert test.prediction_time.min()>=T1 and refit.prediction_time.max()<T1 and refit.resolved_at.max()<T1
assert test.prediction_time.min()>refit.prediction_time.max()
for frame in [train,val,test,refit]:
    assert frame.conditional_profit_label.isin([0,1]).all()
    if frame.feature_week_history_latest_time.notna().any():
        assert (frame.loc[frame.feature_week_history_latest_time.notna(),'feature_week_history_latest_time']<frame.loc[frame.feature_week_history_latest_time.notna(),'prediction_time']).all()
cohort_summary=[]
for name,z in [('all_input',d),('train',train),('validation',val),('test',test),('final_refit',refit)]:
    cohort_summary.append({'cohort':name,'rows':len(z),'wallets':z.wallet.nunique(),'markets':z.market.nunique(),
      'event_proxies':z.event_group.nunique(),'labelled':int(z.conditional_profit_label.notna().sum()),
      'positive_rate_conditional':z.conditional_profit_label.mean(),
      'first_entry':z.prediction_time.min(),'last_entry':z.prediction_time.max(),'last_recorded_resolution':z.resolved_at.max()})
cohort_summary=pd.DataFrame(cohort_summary);display(cohort_summary);cohort_summary.to_csv(OUT/'cohort_summary.csv',index=False)
event_audit=d.groupby('event_group').agg(markets=('market','nunique'),rows=('wallet','size'),
 first_entry=('prediction_time','min'),last_entry=('prediction_time','max'),last_recorded_resolution=('resolved_at','max'),
 assigned_partition=('assigned_partition','first'),event_rule=('event_mapping_rule','first'))
events.to_csv(OUT/'event_mapping.csv',index=False);event_audit.to_csv(OUT/'event_split_audit.csv')
d[['wallet','market','prediction_time','resolved_at','event_group','assigned_partition','eligible_conditional','split','final_refit','exclusion_reason']].to_csv(OUT/'row_split_audit.csv',index=False)
d.groupby('exclusion_reason').size().rename('rows').to_csv(OUT/'exclusion_counts.csv')
print('测试集中与最终训练集相同钱包的比例:',test.wallet.isin(refit.wallet).mean())
print('所有事件组为透明规则代理，不是经核验的真实事件ID；真实结果公开时间仍未知。')



## 复用已验证的协方差与预处理

沿用市场去均值LPM与钱包+事件两维聚类，有限样本校正计入吸收的市场自由度，t/F参照df=min钱包簇/事件簇−1；小fixture与完整dummy OLS一致。整体非正定协方差不修补，联合子块不正定或奇异则检验不可用。所有缺失填补/缩放只在对应训练行拟合。

In [ ]:
def raw_design(frame,history=True,association=False):
    X=pd.DataFrame(index=frame.index)
    X['entry_log_buy']=frame.feature_entry_log_buy_dollars
    p=frame.feature_entry_buy_vwap
    X['entry_price']=p;X['entry_price_squared']=p**2;X['entry_price_cubed']=p**3
    if not association:X['entry_concentration']=frame.feature_entry_direction_concentration
    X['entry_log_fill_rows']=np.log1p(frame.feature_entry_fill_rows)
    X['market_log_dollars']=frame.feature_strict_prior_market_log_dollars
    X['market_log_wallets']=frame.feature_strict_prior_market_log_wallets
    X['market_log_observed_age']=np.log1p(frame.feature_observed_market_age_hours.clip(lower=0))
    hour=frame.feature_entry_utc_hour
    X['clock_sin']=np.sin(2*np.pi*hour/24);X['clock_cos']=np.cos(2*np.pi*hour/24)
    X['window_exposure_days']=(frame.prediction_time-pd.Timestamp('2026-09-29',tz='UTC')).dt.total_seconds()/86400
    if history:
        X['history_log_fills']=frame.feature_week_history_log_fill_rows
        X['history_log_markets']=frame.feature_week_history_log_markets
        X['history_log_buy']=frame.feature_week_history_log_buy_dollars
        X['history_buy_hhi']=frame.feature_week_history_buy_market_hhi
        X['history_log_span']=np.log1p(frame.feature_week_history_observed_span_hours.clip(lower=0))
        X['history_log_recency']=frame.feature_week_history_log_recency_hours
    for name in ['entry_price','history_buy_hhi','history_log_span','history_log_recency']:
        if name in X:X[name+'__missing']=X[name].isna().astype(float)
    assert not any(re.search(r'label|audit|payout|cashflow|final',c) for c in X.columns)
    return X.astype(float)

class TrainPreprocessor:
    def fit(self,X):
        self.columns=list(X);self.median=X.median().fillna(0.)
        filled=X.fillna(self.median)
        self.mean=filled.mean();self.scale=filled.std(ddof=0).replace(0,1.)
        for c in X:
            if c.endswith('__missing'):self.mean[c]=0.;self.scale[c]=1.
        self.fit_rows=len(X)
        return self
    def transform(self,X):
        assert list(X)==self.columns
        return ((X.fillna(self.median)-self.mean)/self.scale).to_numpy(dtype=float)

def within(A,group):
    return A-pd.DataFrame(A).groupby(np.asarray(group),sort=False).transform('mean').to_numpy()

def cluster_meat(scores,groups):
    codes,_=pd.factorize(pd.Index(groups,tupleize_cols=False),sort=False)
    grouped=np.zeros((codes.max()+1,scores.shape[1]));np.add.at(grouped,codes,scores)
    return grouped.T@grouped,len(grouped)

def covariance_2way(X,resid,wallet,event,absorbed_groups):
    n,k=X.shape;bread=np.linalg.inv(X.T@X);scores=X*resid[:,None]
    terms=[];counts=[]
    for groups in [wallet,event,list(zip(wallet,event))]:
        meat,G=cluster_meat(scores,groups)
        if G<2 or n-k-absorbed_groups<=0:raise ValueError('Insufficient clusters or residual degrees of freedom')
        correction=G/(G-1)*(n-1)/(n-k-absorbed_groups)
        terms.append(correction*bread@meat@bread);counts.append(G)
    C=terms[0]+terms[1]-terms[2]
    return (C+C.T)/2,counts

def fit_logistic(X,y,label):
    model=LogisticRegression(C=CONFIG['logistic_C'],max_iter=CONFIG['logistic_max_iter'],solver='lbfgs',random_state=CONFIG['seed'])
    with warnings.catch_warnings(record=True) as ws:
        warnings.simplefilter('always');model.fit(X,np.asarray(y,dtype=int))
    MODEL_CHECKS.append({'model':label,'iterations':int(model.n_iter_.max()),'converged':bool(model.n_iter_.max()<CONFIG['logistic_max_iter']),
                         'warnings':';'.join(str(w.message) for w in ws),'rows':len(y),'features':X.shape[1]})
    return model

def probability_metrics(y,p):
    y=np.asarray(y,dtype=int);p=np.clip(np.asarray(p),1e-12,1-1e-12)
    binary=(p>=CONFIG['classification_threshold']).astype(int)
    return {'rows':len(y),'positive_rate':float(y.mean()),'log_loss':log_loss(y,p,labels=[0,1]),
       'brier':brier_score_loss(y,p),'roc_auc':roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan,
       'pr_auc_average_precision':average_precision_score(y,p) if y.sum()>0 else np.nan,
       'mean_prediction':float(p.mean()),'calibration_mean_error':float(p.mean()-y.mean()),
       'accuracy_threshold_0_5':float((binary==y).mean())}

def paired_event_delta(y,reference,alternative,events,seed=20261019):
    # Positive alternative-minus-reference loss means the reference model helped.
    y=np.asarray(y,dtype=float);r=np.clip(reference,1e-12,1-1e-12);a=np.clip(alternative,1e-12,1-1e-12)
    frame=pd.DataFrame({'event':np.asarray(events),'n':1,
      'log_loss_delta':-(y*np.log(a)+(1-y)*np.log(1-a))+(y*np.log(r)+(1-y)*np.log(1-r)),
      'brier_delta':(y-a)**2-(y-r)**2})
    grouped=frame.groupby('event').sum();G=len(grouped);rng=np.random.default_rng(seed)
    draws=rng.integers(0,G,size=(CONFIG['bootstrap_replicates'],G))
    answer={'event_proxy_groups':G,'bootstrap_replicates':CONFIG['bootstrap_replicates'],
      'effective_event_proxy_groups_inverse_hhi':float(1/((grouped.n/grouped.n.sum())**2).sum()),
      'largest_event_row_share':float(grouped.n.max()/grouped.n.sum())}
    for c in ['log_loss_delta','brier_delta']:
        boot=grouped[c].to_numpy()[draws].sum(axis=1)/grouped.n.to_numpy()[draws].sum(axis=1)
        answer[c]=float(frame[c].mean());answer[c+'_ci_low']=float(np.quantile(boot,.025));answer[c+'_ci_high']=float(np.quantile(boot,.975))
    return answer

# Small covariance fixture: absorbed slopes match full dummy OLS, including absorbed FE df.
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_cluster_2groups
rng=np.random.default_rng(12);n=160;g=np.repeat(np.arange(8),20);wa=np.tile(np.arange(20),8);ev=g//2
xf=rng.normal(size=(n,2));yf=xf@np.array([.3,-.2])+g*.1+rng.normal(size=n)
xw=within(xf,g);yw=within(yf[:,None],g).ravel();bf=np.linalg.lstsq(xw,yw,rcond=None)[0]
cf,_=covariance_2way(xw,yw-xw@bf,wa,ev,len(np.unique(g)))
dummies=pd.get_dummies(g,dtype=float).to_numpy();fixture_fit=sm.OLS(yf,np.column_stack([xf,dummies])).fit()
refcov=cov_cluster_2groups(fixture_fit,wa,ev,use_correction=True)[0][:2,:2]
assert np.allclose(bf,fixture_fit.params[:2],atol=1e-10)
assert np.allclose(cf,refcov,atol=1e-10)
assert np.allclose(within(xf,g).reshape(8,20,2).mean(axis=1),0,atol=1e-12)
print('Within-market and two-way covariance/dummy-OLS fixtures passed.')

# All later analyses use the same complete provided-file24h cohort; early rows remain in feature exports.
cohort_before=[]
for label,frame in [('train',train),('validation',val),('test_reused',test),('final_refit',refit)]:
 cohort_before.append({'split':label,'original_rows':len(frame),'full24h_rows':int(frame.audit_only_fixed24h_left_truncated.eq(0).sum()),
                       'left_truncated_rows':int(frame.audit_only_fixed24h_left_truncated.eq(1).sum())})
train=train[train.audit_only_fixed24h_left_truncated.eq(0)].copy()
val=val[val.audit_only_fixed24h_left_truncated.eq(0)].copy()
test=test[test.audit_only_fixed24h_left_truncated.eq(0)].copy()
refit=refit[refit.audit_only_fixed24h_left_truncated.eq(0)].copy()
cohort_coverage=pd.DataFrame(cohort_before);cohort_coverage.to_csv(OUT/'common_full24h_cohort_coverage.csv',index=False);display(cohort_coverage)
def row_fingerprint(frame):return hashlib.sha256(frame[['wallet','market']].to_csv(index=False).encode()).hexdigest()
common_fingerprints={k:row_fingerprint(z) for k,z in [('train',train),('validation',val),('test_reused',test),('final_refit',refit)]}
assert not set(train.event_group)&set(test.event_group) and not set(refit.market)&set(test.market)
MODEL_CHECKS=[]
# Fail before any model fit if an exact-matched official event ID crosses evaluation partitions.
for left,right in [(train,val),(train,test),(val,test),(refit,test)]:
 left_ids=set(identity for market in left.market.unique() for identity in official_links.get(market,[]))
 right_ids=set(identity for market in right.market.unique() for identity in official_links.get(market,[]))
 assert not left_ids&right_ids,('known_official_event_crosses_model_split',left_ids&right_ids)


## 同一批行：基线、各历史单项、全部三项

基线保持价格三次项、当前规模/填单、市场背景和时间项。先在共同开发训练行拟合一次完整预处理，所有嵌套模型共用同一套基线尺度与行键hash。原值/log值不重复堆入。累计周和24h分别做3个单项模型与1个全三项模型，外加共同基线。冻结8项Holm族=6个单项检验+2个全三项联合检验；全三项中的6个单系数为另一个探索性族。

报告系数变动和条件数，但它们不能证明“共线性造成不显著”或经验没有总价值。LPM是条件关联，不是因果技能/净利润。

In [ ]:
EXPERIENCE=['log_fills','log_markets','log_buy_dollars']
def experience_design(frame,period):
 source='feature_week_history_' if period=='expanding' else 'feature_fixed24h_'
 return pd.DataFrame({'log_fills':frame[source+'log_fill_rows'],
   'log_markets':frame[source+'log_markets'],'log_buy_dollars':frame[source+'log_buy_dollars']},index=frame.index)
base=raw_design(train,history=False,association=True)
full_inference=base.copy()
for period in ['expanding','fixed24h']:
 for c in EXPERIENCE:full_inference[period+'__'+c]=experience_design(train,period)[c]
common_pre=TrainPreprocessor().fit(full_inference)
common_matrix=common_pre.transform(full_inference);common_names=list(full_inference)
y=train.conditional_profit_label.to_numpy(dtype=float);market=train.market.to_numpy();markets=train.market.nunique()
Xwithin=within(common_matrix,market);ywithin=within(y[:,None],market).ravel()

def nested_fit(selected,label):
 keep=[];dropped=[];rank=0
 for c in selected:
  j=common_names.index(c);newrank=np.linalg.matrix_rank(Xwithin[:,keep+[j]],tol=1e-8)
  if np.linalg.norm(Xwithin[:,j])<=1e-8 or newrank<=rank:dropped.append(c)
  else:keep.append(j);rank=newrank
 names=[common_names[j] for j in keep];X=Xwithin[:,keep]
 beta=np.linalg.lstsq(X,ywithin,rcond=None)[0];residual=ywithin-X@beta
 cov,clusters=covariance_2way(X,residual,train.wallet.to_numpy(),train.event_group.to_numpy(),markets)
 dfref=min(clusters[:2])-1;diag=np.diag(cov);se=np.where(diag>0,np.sqrt(np.maximum(diag,0)),np.nan)
 p=2*stats.t.sf(abs(beta/se),dfref);critical=stats.t.ppf(.975,dfref)
 co=pd.DataFrame({'model':label,'term':names,'coefficient':beta,'se_2way':se,'p_raw':p,
   'ci95_low':beta-critical*se,'ci95_high':beta+critical*se,'effect_percentage_points':100*beta,
   'ci95_low_percentage_points':100*(beta-critical*se),'ci95_high_percentage_points':100*(beta+critical*se),
   'units':['0to1 indicator' if c.endswith('__missing') else 'one common-development-cohort SD' for c in names]})
 sizes=train.event_group.value_counts(normalize=True)
 diagnostics={'model':label,'rows':len(train),'row_fingerprint':row_fingerprint(train),'markets':markets,
  'slope_rank':len(names),'selected_terms':len(selected),'dropped_terms':';'.join(dropped),
  'within_condition_number':float(np.linalg.cond(X)),'wallet_clusters':clusters[0],'event_proxy_clusters':clusters[1],
  'event_effective_inverse_hhi':float(1/(sizes*sizes).sum()),'largest_event_share':float(sizes.max()),
  'df_reference':dfref,'residual_df':len(train)-markets-len(names),
  'covariance_rank':int(np.linalg.matrix_rank(cov)),'covariance_min_eigenvalue':float(np.linalg.eigvalsh(cov).min())}
 return co,cov,names,beta,diagnostics

def wald_test(result,terms):
 co,cov,names,beta,di=result
 if any(c not in names for c in terms):return {'p_raw':np.nan,'test_status':'unavailable_aliased_required_term','df_num':len(terms)}
 idx=[names.index(c) for c in terms];sub=cov[np.ix_(idx,idx)];eig=np.linalg.eigvalsh(sub)
 if np.linalg.matrix_rank(sub)<len(terms) or eig.min()<=1e-12:
  return {'p_raw':np.nan,'test_status':'unavailable_nonpositive_or_singular_covariance','df_num':len(terms)}
 F=float(beta[idx]@np.linalg.solve(sub,beta[idx])/len(terms))
 return {'p_raw':float(stats.f.sf(F,len(terms),di['df_reference'])),'F':F,'test_status':'estimable','df_num':len(terms),'df_reference':di['df_reference']}
results={};coefficients=[];diagnostics=[];frozen_tests=[]
baseline_result=nested_fit(list(base),'baseline');results['baseline']=baseline_result
coefficients.append(baseline_result[0]);diagnostics.append(baseline_result[4])
for period in ['expanding','fixed24h']:
 for term in EXPERIENCE:
  label=period+'_only_'+term;fullterm=period+'__'+term
  fitted=nested_fit(list(base)+[fullterm],label);results[label]=fitted
  coefficients.append(fitted[0]);diagnostics.append(fitted[4])
  frozen_tests.append({'lookback':period,'hypothesis':'single_'+term,'model':label,**wald_test(fitted,[fullterm])})
 label=period+'_all_three';terms=[period+'__'+c for c in EXPERIENCE]
 fitted=nested_fit(list(base)+terms,label);results[label]=fitted
 coefficients.append(fitted[0]);diagnostics.append(fitted[4])
 frozen_tests.append({'lookback':period,'hypothesis':'joint_all_three','model':label,**wald_test(fitted,terms)})
coef=pd.concat(coefficients,ignore_index=True);nested_diagnostics=pd.DataFrame(diagnostics);tests=pd.DataFrame(frozen_tests)
assert len(tests)==8 and nested_diagnostics.row_fingerprint.nunique()==1
assert nested_diagnostics.row_fingerprint.iloc[0]==common_fingerprints['train']
tests['p_holm_frozen_8_test_family']=multipletests(tests.p_raw.fillna(1.),method='holm')[1]
tests.loc[tests.p_raw.isna(),'p_holm_frozen_8_test_family']=np.nan
movements=[]
for period in ['expanding','fixed24h']:
 for c in EXPERIENCE:
  term=period+'__'+c;single=coef[coef.model.eq(period+'_only_'+c)&coef.term.eq(term)]
  multiple=coef[coef.model.eq(period+'_all_three')&coef.term.eq(term)]
  if len(single)==1 and len(multiple)==1:
   a=single.iloc[0];b=multiple.iloc[0]
   movements.append({'lookback':period,'feature':c,'single_coefficient':a.coefficient,'all_three_coefficient':b.coefficient,
      'single_ci_low':a.ci95_low,'single_ci_high':a.ci95_high,'all_three_ci_low':b.ci95_low,'all_three_ci_high':b.ci95_high,
      'single_p_raw':a.p_raw,'all_three_p_raw_exploratory':b.p_raw})
movements=pd.DataFrame(movements)
# Six all-three conditional individual coefficients form a separate exploratory family.
mask=coef.model.str.endswith('_all_three')&coef.term.str.contains('__')&~coef.term.str.endswith('__missing')
coef['p_holm_all_three_coefficients_exploratory']=np.nan
ok=mask&coef.p_raw.notna()
if ok.any():coef.loc[ok,'p_holm_all_three_coefficients_exploratory']=multipletests(coef.loc[ok,'p_raw'],method='holm')[1]
coef.to_csv(OUT/'nested_fe_coefficients.csv',index=False);tests.to_csv(OUT/'nested_frozen_concept_tests.csv',index=False)
nested_diagnostics.to_csv(OUT/'nested_design_diagnostics.csv',index=False);movements.to_csv(OUT/'nested_coefficient_movements.csv',index=False)
display(tests);display(movements);display(nested_diagnostics)


## 冻结预测对比（测试复用）

固定C=1的logistic、相同当前/背景基线与相同人群，比较基线、累计3项、24h3项、累计5项、24h5项；后两项额外加入HHI和平均BUYticket。价格/基线相同，仅历史定义不同。不按已经看过的测试集调参、选模型、换窗口或阈值。事件bootstrap只描述固定预测，不能重现钱包跨事件依赖。

In [ ]:
# Context/current controls match Round04; only history definitions differ in matched comparisons.
def prediction_design(frame,variant):
 X=raw_design(frame,history=False,association=False)
 if variant=='baseline':return X
 period,size=variant.split('_')
 for c in EXPERIENCE:X['history__'+c]=experience_design(frame,period)[c]
 if size=='5':
  source='feature_week_history_' if period=='expanding' else 'feature_fixed24h_'
  X['history__buy_hhi']=frame[source+'buy_market_hhi']
  X['history__log_avg_buy_ticket']=frame[source+'log_avg_buy_ticket']
  for c in ['history__buy_hhi','history__log_avg_buy_ticket']:X[c+'__missing']=X[c].isna().astype(float)
 return X
variants=['baseline','expanding_3','fixed24h_3','expanding_5','fixed24h_5']
predictions={};prediction_metrics=[];validation_metrics=[];prediction_fingerprints=[]
for variant in variants:
 for training,evaluated,phase in [(train,val,'validation_reused'),(refit,test,'test_reused')]:
  raw=prediction_design(training,variant);pre=TrainPreprocessor().fit(raw)
  model=fit_logistic(pre.transform(raw),training.conditional_profit_label,variant+'_'+phase)
  pred=model.predict_proba(pre.transform(prediction_design(evaluated,variant)))[:,1]
  record={'variant':variant,'evaluation_status':CONFIG['evaluation_status'],**probability_metrics(evaluated.conditional_profit_label,pred)}
  prediction_fingerprints.append({'variant':variant,'phase':phase,'training_rows':len(training),'evaluated_rows':len(evaluated),
      'training_keys_sha256':row_fingerprint(training),'evaluation_keys_sha256':row_fingerprint(evaluated)})
  if phase=='test_reused':predictions[variant]=pred;prediction_metrics.append(record)
  else:validation_metrics.append(record)
metric_table=pd.DataFrame(prediction_metrics);metric_table.to_csv(OUT/'exploratory_reused_test_metrics.csv',index=False)
pd.DataFrame(validation_metrics).to_csv(OUT/'exploratory_reused_validation_metrics.csv',index=False)
fingerprints=pd.DataFrame(prediction_fingerprints)
assert fingerprints.groupby('phase').training_keys_sha256.nunique().eq(1).all()
assert fingerprints.groupby('phase').evaluation_keys_sha256.nunique().eq(1).all()
fingerprints.to_csv(OUT/'predictive_common_cohort_fingerprints.csv',index=False)
prediction_export=test[['wallet','market','prediction_time','event_group','conditional_profit_label']].copy()
for variant,pred in predictions.items():prediction_export[variant]=pred
prediction_export.to_csv(OUT/'exploratory_reused_test_predictions.csv',index=False)
deltas=[]
for reference,alternative in [('expanding_3','baseline'),('fixed24h_3','baseline'),('fixed24h_3','expanding_3'),
                              ('expanding_5','baseline'),('fixed24h_5','baseline'),('fixed24h_5','expanding_5')]:
 deltas.append({'reference':reference,'alternative':alternative,
    'sign_definition':'positive alternative-minus-reference loss means reference is better; exploratory reused test',
    **paired_event_delta(test.conditional_profit_label,predictions[reference],predictions[alternative],test.event_group)})
deltas=pd.DataFrame(deltas);deltas.to_csv(OUT/'exploratory_paired_loss_deltas.csv',index=False)
print('以下是已经看过的Round04测试时段的探索性复用，不是新鲜独立测试；模型、24h窗口和特征集合均未按结果调优。')
display(metric_table);display(deltas)


## 证据覆盖、实际结果与边界

下方实际数值解读覆盖链上映射范围、同样本嵌套系数/联合检验、完整24h人群和复用测试损失。代理事件集中会让bootstrap区间不稳定；这些不是新鲜总体确认性证据。

方法参考：[statsmodels两维聚类](https://www.statsmodels.org/stable/generated/statsmodels.stats.sandwich_covariance.cov_cluster_2groups.html)、[sklearn避免泄漏](https://scikit-learn.org/stable/common_pitfalls.html)。

In [ ]:
# An independent event ID does not establish independent macro shocks; keep conservative proxy unions.
def identities_for(frame):return set(identity for market in frame.market.unique() for identity in official_links.get(market,[]))
official_split_checks=[]
for left_name,left,right_name,right in [('train',train,'validation',val),('train',train,'test',test),('validation',val,'test',test),('refit',refit,'test',test)]:
 overlap=identities_for(left)&identities_for(right)
 official_split_checks.append({'left':left_name,'right':right_name,'shared_matched_official_event_ids':len(overlap),
  'left_matched_ids':len(identities_for(left)),'right_matched_ids':len(identities_for(right))})
 assert not overlap,('official_event_leakage',left_name,right_name,overlap)
pd.DataFrame(official_split_checks).to_csv(OUT/'official_event_split_disjointness_checks.csv',index=False)
sizes=test.event_group.value_counts();shares=sizes/sizes.sum()
event_diagnostics={'event_proxy_groups':len(sizes),'effective_inverse_hhi':float(1/(shares*shares).sum()),
 'largest_group_share':float(shares.max()),'largest_group':str(sizes.index[0]),
 'bootstrap_limitation':'Event-only resampling does not resample cross-event repeated-wallet dependence; highly concentrated groups make intervals unstable/descriptive.'}
Path(OUT/'event_concentration_diagnostics.json').write_text(json.dumps(event_diagnostics,indent=2))
# Mapping scope and contradictions; do not silently recode any model target.
verified=mapping_evidence['onchain_verified'].map(as_bool) if 'onchain_verified' in mapping_evidence else pd.Series(False,index=mapping_evidence.index)
matched=mapping_evidence['base_mapping_matches'].map(as_bool) if 'base_mapping_matches' in mapping_evidence else pd.Series(False,index=mapping_evidence.index)
verified_markets=set(mapping_evidence.loc[verified,'market']) if 'market' in mapping_evidence else set()
contradictions=mapping_evidence[verified&~matched] if 'base_mapping_matches' in mapping_evidence else mapping_evidence.iloc[:0]
contradictions.to_csv(OUT/'verified_mapping_base_contradictions.csv',index=False)
mapping_summary={'evidence_rows':len(mapping_evidence),'verified_market_count':len(verified_markets),
 'verified_base_disagreements':len(contradictions),'candidate_markets':d.market.nunique(),
 'candidate_rows_with_verified_mapping':int(d.market.isin(verified_markets).sum()),
 'labels_rechosen_using_model_fit':False,'main_label_mapping_changed':False}
Path(OUT/'mapping_evidence_summary.json').write_text(json.dumps(mapping_summary,indent=2))
# Units/definitions for newly constructed fields.
window_dictionary=[]
for c in [c for c in d if c.startswith('feature_fixed24h_') or c in ['feature_week_history_buy_fill_rows','feature_week_history_avg_buy_ticket','feature_week_history_log_avg_buy_ticket']]:
 units='unitless' if 'log_' in c or 'hhi' in c or 'no_' in c else ('recorded dollars per BUYfill' if 'avg_buy_ticket' in c else ('recorded dollars' if 'dollars' in c else 'count'))
 window_dictionary.append({'column':c,'units':units,'definition':c.replace('feature_','').replace('_',' '),
  'time_rule':'[T-24h,T), whole target-market excluded; full coverage T>=Sep30' if 'fixed24h' in c else 'supplied-week strict-prior other-market history',
  'missingness':'Full24h fields unavailable before coverage boundary; HHI missing if no BUY dollars; average ticket missing if no BUY rows (zero-dollar BUY rows count)'})
pd.DataFrame(window_dictionary).to_csv(OUT/'fixed_window_feature_dictionary.csv',index=False)
pd.DataFrame(MODEL_CHECKS).to_csv(OUT/'model_convergence_checks.csv',index=False)
text=['# 本轮实际结果：结算映射与历史窗口诊断',
 '**Round04测试时段已经看过，本轮所有预测比较均为探索性复用，不是新的独立测试；固定24h、模型和检验族没有按p值或测试效果选择。**',
 f"映射证据表 {mapping_summary['evidence_rows']} 行；独立链上验证覆盖 {mapping_summary['verified_market_count']}/{mapping_summary['candidate_markets']} 个候选市场、{mapping_summary['candidate_rows_with_verified_mapping']:,} 个输入组合。已验证但与主映射矛盾的市场数 {mapping_summary['verified_base_disagreements']}。未覆盖部分仍未核验；即使结算正确，费用/期初持仓/非交易变化缺失仍使真实净利润未知。本轮没有按拟合效果挑映射，也没有静默改标签。",
 f"事件链接使用 {official_linkage_summary['identity_field_used']} 为真时的官方ID，将保守组只合并不拆细：{official_linkage_summary['original_proxy_groups']}→{official_linkage_summary['official_connected_proxy_groups']} 组。主比较在同一官方链接兼容人群上进行；独立官方ID跨分割重叠检查全部通过。未匹配ID不等于已证明无关联。",
 '## 同一批行上的嵌套关联',
 f"所有嵌套模型使用完全相同 {len(train):,} 行，键SHA256={common_fingerprints['train']}；价格三次项/市场背景/时间控制、市场FE和基线预处理固定。下列8项构成预先冻结Holm检验族，无法估计的项保留族大小但不填假p值。"]
zh={'log_fills':'历史成交行数','log_markets':'历史市场数','log_buy_dollars':'历史买入金额'}
for row in tests.itertuples(index=False):
 if not np.isfinite(row.p_raw):reading='无法给出有效检验，不等于没有关联'
 elif row.p_holm_frozen_8_test_family<.05:reading='校正后有条件关联证据'
 else:reading='校正后没有清晰条件关联证据，不等于经验没有总价值'
 text.append(f'- {row.lookback} / {row.hypothesis}：原始p={row.p_raw:.3g}，8项Holm p={row.p_holm_frozen_8_test_family:.3g}；{reading}。')
for row in movements.itertuples(index=False):
 text.append(f'- {row.lookback} / {zh[row.feature]}：单独加入基线β={row.single_coefficient:+.4f}，与其他两项同时加入β={row.all_three_coefficient:+.4f}；后者95%CI=[{row.all_three_ci_low:+.4f},{row.all_three_ci_high:+.4f}]（共同训练标准差单位）。')
text+=['系数变化与宽区间可以显示条件比较的不稳定，但不能单凭它断言共线性“造成”此前不显著，也不能把正/负系数当因果钱包能力。',
 '## 严格24h与累计周历史的匹配预测',
 f"所有方案最终训练 {len(refit):,} 行、复用测试 {len(test):,} 行；测试事件组 {event_diagnostics['event_proxy_groups']}，有效组数约{event_diagnostics['effective_inverse_hhi']:.2f}，最大组占{event_diagnostics['largest_group_share']:.2%}。事件bootstrap没有重采样钱包跨事件依赖，区间仅描述当前固定模型在代理独立假设下的波动，不能作确认性显著结论。"]
for row in metric_table.itertuples(index=False):text.append(f'- {row.variant}：log loss={row.log_loss:.5f}，Brier={row.brier:.5f}，ROC-AUC={row.roc_auc:.4f}，PR-AUC={row.pr_auc_average_precision:.4f}。')
for row in deltas.itertuples(index=False):
 reading='区间跨0，未见清晰差异' if row.log_loss_delta_ci_low<=0<=row.log_loss_delta_ci_high else '区间不跨0，但仍是复用测试的描述性比较，不用于选择方案'
 text.append(f'- {row.alternative}−{row.reference} Δlog loss={row.log_loss_delta:+.6f}，事件bootstrap区间[{row.log_loss_delta_ci_low:+.6f},{row.log_loss_delta_ci_high:+.6f}]；{reading}。正值表示reference损失更低。')
text+=['## 边界与下一步','24h下界包括、上界整批排除，历史市场数包含买卖，平均BUYticket按BUY行数计算，0美元BUY仍是有效BUY行；无BUY美元时HHI缺失。Sep30前的不足24h行在特征表明确标记，未补0混入主比较。',
 '本周历史不是终身经验/财富；候选宇宙本身事后选择。需要真实结果公开/可交易状态、完整费用/期初成本/非交易持仓变化，以及更长时间或未来新测试段，才能改善部署和总体推断。当前结果不用于追逐显著、删除“不显著”特征或宣称真实净利润预测。']
if MODE=='smoke':text.insert(1,'**SMOKE运行验证，不作为本轮最终科学结果。**')
interpretation='\n\n'.join(text);(OUT/'interpretation_zh.md').write_text(interpretation);display(Markdown(interpretation))
provenance.update({'window_diagnostics':window_diagnostics,'independent_verification':verification_report,
 'common_cohort_fingerprints':common_fingerprints,'mapping_summary':mapping_summary,
 'official_linkage_summary':official_linkage_summary,'event_concentration':event_diagnostics,
 'elapsed_seconds':time.perf_counter()-START,
 'output_hashes':{p.name:sha(p) for p in sorted(OUT.iterdir()) if p.is_file() and p.name!='validation_provenance.json'}})
Path(OUT/'validation_provenance.json').write_text(json.dumps(provenance,indent=2,default=str))
print('本轮所有计算完成，秒:',round(time.perf_counter()-START,2))
